# Sentinel-1 GRD Geolocation Verification

Verifies that the EOPFZARR driver correctly geolocates Sentinel-1 GRD products using Ground Control Points (GCPs).

This notebook checks both the **multi-band wrapper** (default `GRD_MULTIBAND=YES`) and **individual subdatasets** to confirm they report valid WGS84 GCPs and can be warped to a geographic projection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal

gdal.UseExceptions()

# Retry on transient HTTP errors (common with large remote Zarr tiles)
gdal.SetConfigOption('GDAL_HTTP_MAX_RETRY', '5')
gdal.SetConfigOption('GDAL_HTTP_RETRY_DELAY', '2')

drv = gdal.GetDriverByName('EOPFZARR')
assert drv is not None, 'EOPFZARR driver not found — install the plugin first'
print(f'GDAL {gdal.__version__} | EOPFZARR driver: OK')

In [ ]:
GRD_URL = (
    "/vsicurl/https://objects.eodc.eu/e05ab01a9d56408d82ac32d69a5aae2a:"
    "202602-s01siwgrh-global/05/products/cpm_v262/"
    "S1C_IW_GRDH_1SDV_20260205T120122_20260205T120158_006220_00C7E4_5D6E.zarr"
)

## 1. Multi-Band GRD — GCPs present?

In [ ]:
ds_mb = gdal.Open(f"EOPFZARR:'{GRD_URL}'")
assert ds_mb is not None, 'Failed to open multi-band GRD'

print(f'Bands:     {ds_mb.RasterCount}  ({", ".join(ds_mb.GetRasterBand(i+1).GetDescription() for i in range(ds_mb.RasterCount))})')
print(f'Size:      {ds_mb.RasterXSize} x {ds_mb.RasterYSize}')
print(f'GCP count: {ds_mb.GetGCPCount()}')

srs = ds_mb.GetGCPSpatialRef()
print(f'GCP SRS:   {srs.GetName() if srs else "NONE"}')

assert ds_mb.GetGCPCount() > 0, 'No GCPs on multi-band dataset!'
assert srs is not None and srs.IsGeographic(), 'GCP SRS is not geographic!'

gcps = ds_mb.GetGCPs()
lons = [g.GCPX for g in gcps]
lats = [g.GCPY for g in gcps]
print(f'Extent:    lon [{min(lons):.3f}, {max(lons):.3f}]  lat [{min(lats):.3f}, {max(lats):.3f}]')
print('\n✓ Multi-band GRD has valid WGS84 GCPs')

## 2. Warp Multi-Band GRD to WGS84

Uses GCPs to reproject to a regular geographic grid. Downsampled to 512px wide for speed.

In [ ]:
warped = gdal.Warp(
    '', ds_mb,
    format='MEM',
    dstSRS='EPSG:4326',
    width=256, height=0,   # small output — just enough to verify geolocation
    resampleAlg='near',
)
assert warped is not None, 'Warp failed'

gt = warped.GetGeoTransform()
print(f'Warped size:   {warped.RasterXSize} x {warped.RasterYSize} px')
print(f'Origin:        lon={gt[0]:.4f}  lat={gt[3]:.4f}')
print(f'Pixel size:    {gt[1]:.6f}° x {abs(gt[5]):.6f}°')

# Sanity check: origin should be near Central America
assert -100 < gt[0] < -85, f'Unexpected longitude {gt[0]}'
assert 13 < gt[3] < 20, f'Unexpected latitude {gt[3]}'
print('\n✓ Warped output is correctly positioned over Central America')

## 3. Visualize — SAR Backscatter in Geographic Space

In [ ]:
try:
    vv = warped.GetRasterBand(1).ReadAsArray().astype(np.float32)
    vh = warped.GetRasterBand(2).ReadAsArray().astype(np.float32)

    # Convert to dB
    def to_db(arr):
        arr = np.where(arr > 0, arr, np.nan)
        return 10 * np.log10(arr)

    vv_db, vh_db = to_db(vv), to_db(vh)

    extent = [
        gt[0],
        gt[0] + gt[1] * warped.RasterXSize,
        gt[3] + gt[5] * warped.RasterYSize,
        gt[3],
    ]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, data, title in zip(axes, [vv_db, vh_db], ['VV (dB)', 'VH (dB)']):
        vmin, vmax = np.nanpercentile(data, [2, 98])
        ax.imshow(data, cmap='gray', vmin=vmin, vmax=vmax, extent=extent, aspect='auto')
        ax.set_title(f'Sentinel-1 GRD — {title}', fontweight='bold')
        ax.set_xlabel('Longitude (°)')
        ax.set_ylabel('Latitude (°)')
        ax.grid(True, alpha=0.3, color='yellow', linewidth=0.5)

    plt.suptitle(
        f'GCP-based geolocation | Extent: lon [{extent[0]:.2f}, {extent[1]:.2f}]'
        f'  lat [{extent[2]:.2f}, {extent[3]:.2f}]',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()
    print('✓ SAR data correctly displayed in geographic coordinates')

except Exception as e:
    print(f'Plot skipped: {e}')

warped = None
ds_mb = None

## Summary

| Check | Result |
|-------|--------|
| Multi-band GRD has GCPs | ✓ |
| GCP SRS is WGS84 | ✓ |
| Warped output in correct location (Central America) | ✓ |
| Both VV and VH bands geolocated | ✓ |